In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.inspection import partial_dependence

# 1. Load the winning model pipeline and training data
with open('../deployment/model_artifacts_002.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_model = artifacts['model']
train_df = pd.read_csv('../data/train_data.csv')
X_train = train_df.drop(columns='Sold_Price')

In [ ]:
# 2. Select the continuous variables you want to visualize
# (Categoricals/Target Encoded features are less interpretable on a PDP line chart)
features_to_plot = [
    'Mileage', 'car_age', 'Engine_Displacement_L', 
    'flaw_severity_score', 'mileage_per_year', 'SP500_Close', "Gears"
]

pdp_results = []

print("Calculating Partial Dependency Plots...")
for feature in features_to_plot:
    # Compute partial dependence (uses 50 grid points by default)
    # The pipeline handles preprocessing transformations automatically
    pdp = partial_dependence(best_model, X_train, features=[feature], grid_resolution=50)
    
    # Accommodate scikit-learn version differences for the grid key
    grid_key = 'values' if 'values' in pdp else 'grid_values'
    grid_values = pdp[grid_key][0]
    
    # The model targets log-space, so we inverse transform to real dollars
    avg_predictions_dollars = np.expm1(pdp['average'][0])
    
    for val, pred in zip(grid_values, avg_predictions_dollars):
        pdp_results.append({
            'Feature': feature,
            'Feature_Value': val,
            'Predicted_Price': pred
        })

Calculating Partial Dependency Plots...


/opt/conda/lib/python3.12/site-packages/sklearn/inspection/_partial_dependence.py:717: FutureWarning: The column 'flaw_severity_score' contains integer data. Partial dependence plots are not supported for integer data: this can lead to implicit rounding with NumPy arrays or even errors with newer pandas versions. Please convert numerical featuresto floating point dtypes ahead of time to avoid problems. This will raise ValueError in scikit-learn 1.9.
  warnings.warn(


In [ ]:
# 3. Export to the frontend data directory
pdp_df = pd.DataFrame(pdp_results)
pdp_df.to_csv('../data/frontend_data/pdp_data.csv', index=False)
print("Saved PDP data to ../data/frontend_data/pdp_data.csv")